In [1]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Import data


In [2]:
# read data
df = pd.read_csv('2018-06-06-ss.cleaned.csv')

In [3]:
df.shape

(393732, 7)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 393732 entries, 0 to 393731
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   pdb_id         393732 non-null  object
 1   chain_code     393732 non-null  object
 2   seq            393732 non-null  object
 3   sst8           393732 non-null  object
 4   sst3           393732 non-null  object
 5   len            393732 non-null  int64 
 6   has_nonstd_aa  393732 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 18.4+ MB


Description of columns

1. pdb_id: the id used to locate its entry on https://www.rcsb.org/
2. chain_code: when a protein consists of multiple peptides (chains), the chain code is needed to locate a particular one.
3. seq: the sequence of the peptide
4. sst8: the eight-state (Q8) secondary structure
5. sst3: the three-state (Q3) secondary structure
6. len: the length of the peptide
7. has_nonstd_aa: whether the peptide contains nonstandard amino acids (B, O, U, X, or Z).

1. Preprocessing data

In [5]:
df = df[df['has_nonstd_aa'] == False]

In [6]:
df_sampled, _ = train_test_split(df, test_size=0.9, random_state=42)

In [7]:
df_sampled.shape

(38633, 7)

In [8]:
#Encoding
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_int = {aa: i for i, aa in enumerate(amino_acids)}

def encode_sequence(seq):
    return [aa_to_int[aa] for aa in seq]

df_sampled['encoded_seq'] = df_sampled['seq'].apply(encode_sequence)

In [9]:
max_seq_len = df_sampled['len'].max()  # or use a fixed value
# Convert sequences to PyTorch tensors
encoded_tensors = [torch.tensor(seq) for seq in df_sampled['encoded_seq']]

In [10]:
max_seq_len

4646

In [11]:
# Pad sequences
padded_seq = nn.utils.rnn.pad_sequence(encoded_tensors, batch_first=True)

In [12]:
encoded_tensors

[tensor([ 3, 11,  7,  2,  7, 16, 17, 15,  0,  0, 16,  9, 15, 15,  7, 15,  7, 15,
         12,  7, 11, 16, 11,  7, 11, 16, 16, 17, 15,  8, 13,  4,  4,  0, 17,  5,
         16, 19, 15,  2,  5, 16,  8,  0,  2,  9, 16, 15, 15, 17, 16, 18, 15, 15,
         15, 11, 13, 15, 13,  0,  8, 17, 15, 11,  0, 15,  3, 16,  8,  5,  9, 17,
         16,  5,  7,  0, 15,  5, 11, 12, 16,  7,  7,  0, 16, 19,  5, 15, 17, 15,
          5, 11, 16,  7,  9, 16, 17, 11,  8, 16,  2, 16]),
 tensor([ 0, 16, 16,  7, 16, 15, 11, 13, 16,  5, 16,  6,  2,  5, 19,  2, 19,  3,
          9, 18,  8,  2, 15,  5, 11, 16, 15, 10, 16,  9, 11, 15,  5,  5,  0,  4,
         15,  0, 13, 18, 15, 11,  7,  5, 11,  0,  9,  4, 14,  8,  5,  8,  8,  4,
          2, 15, 16,  8, 16,  6, 15, 13,  9,  5, 11,  7, 15,  7, 11, 19, 11,  0,
         16,  4, 11, 12,  5,  5, 11, 15, 19,  9,  1, 17, 19,  5, 18, 16,  8,  2,
         12,  9, 16,  3, 19, 19,  7, 17,  2, 11, 18,  5, 16, 19, 14, 12, 16,  5,
         16, 12,  8,  5, 16,  4, 16, 17,  2,  5,  

In [13]:
padded_seq

tensor([[ 3, 11,  7,  ...,  0,  0,  0],
        [ 0, 16, 16,  ...,  0,  0,  0],
        [10, 15,  6,  ...,  0,  0,  0],
        ...,
        [10,  8,  3,  ...,  0,  0,  0],
        [10,  7,  7,  ...,  0,  0,  0],
        [10,  8, 16,  ...,  0,  0,  0]])

Mapping sst8:
For Q8 (sst8) secondary structure elements:

1. H: Alpha-helix
2. E: Beta-strand
3. C: Coil (random coil or other)
4. G: 3-10 helix
5. I: Pi-helix
6. B: Beta-bridge
7. T: Turn
8. S: Bend

In [14]:
sst8_mapping = {
    'H': 0,  # Alpha-helix
    'E': 1,  # Beta-strand
    'C': 2,  # Coil (Other)
    'G': 3,  # 3-10 helix
    'I': 4,  # Pi-helix
    'B': 5,  # Beta-bridge
    'T': 6,  # Turn
    'S': 7   # Bend
}

In [15]:
def encode_sst8(seq, mapping):
    return [mapping[char] for char in seq if char in mapping]

df_sampled['encoded_sst8'] = df_sampled['sst8'].apply(lambda x: encode_sst8(x, sst8_mapping))

In [16]:
df_sampled['encoded_sst8']

63413     [2, 2, 2, 2, 2, 2, 7, 2, 2, 2, 7, 1, 1, 1, 1, ...
172711    [2, 1, 1, 1, 2, 7, 7, 1, 1, 1, 1, 1, 6, 6, 1, ...
236892    [2, 2, 2, 2, 2, 2, 7, 7, 6, 6, 6, 7, 6, 6, 6, ...
162672    [2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, ...
389088    [2, 2, 2, 6, 6, 6, 6, 7, 2, 7, 7, 2, 2, 2, 6, ...
                                ...                        
265392    [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, ...
372823    [2, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, ...
137589    [2, 2, 2, 2, 7, 1, 1, 1, 1, 2, 2, 6, 6, 6, 6, ...
152583    [2, 2, 1, 1, 1, 1, 6, 6, 6, 2, 2, 1, 1, 1, 1, ...
127567    [2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, ...
Name: encoded_sst8, Length: 38633, dtype: object

In [17]:
#labels
encoded_tensors_ss = [torch.tensor(seq) for seq in df_sampled['encoded_sst8']]
padded_sst8 = nn.utils.rnn.pad_sequence(encoded_tensors_ss, batch_first=True, padding_value=-1)  # Use -1 for padding

In [18]:
encoded_tensors_ss

[tensor([2, 2, 2, 2, 2, 2, 7, 2, 2, 2, 7, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1,
         1, 7, 7, 2, 2, 5, 2, 2, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 2, 2, 7, 7,
         7, 7, 1, 1, 1, 1, 7, 7, 6, 6, 6, 2, 1, 1, 2, 6, 6, 7, 7, 7, 2, 2, 2, 1,
         1, 2, 7, 7, 2, 5, 2, 2, 1, 1, 1, 1, 1, 1, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 2, 7, 2, 2]),
 tensor([2, 1, 1, 1, 2, 7, 7, 1, 1, 1, 1, 1, 6, 6, 1, 1, 1, 1, 1, 1, 1, 7, 7, 7,
         1, 1, 1, 1, 1, 1, 2, 7, 6, 6, 2, 1, 1, 1, 1, 1, 1, 7, 2, 7, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 2, 2, 7, 2, 7, 2, 0, 0, 0, 0, 2, 2, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 2, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 7, 7, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 6, 6, 7, 7, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 7, 7, 2, 2, 7, 1, 1, 1, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6,
         6, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 2, 2, 1, 1, 1, 1,

In [19]:
padded_sst8

tensor([[ 2,  2,  2,  ..., -1, -1, -1],
        [ 2,  1,  1,  ..., -1, -1, -1],
        [ 2,  2,  2,  ..., -1, -1, -1],
        ...,
        [ 2,  2,  2,  ..., -1, -1, -1],
        [ 2,  2,  1,  ..., -1, -1, -1],
        [ 2,  2,  2,  ..., -1, -1, -1]])

In [20]:
# Create a TensorDataset from the padded sequences and labels
dataset = TensorDataset(padded_seq, padded_sst8)

padded_seq_train, padded_seq_test, labels_train, labels_test = train_test_split(
    padded_seq, padded_sst8, test_size=0.2, random_state=42)


train_dataset = TensorDataset(padded_seq_train, labels_train)
test_dataset = TensorDataset(padded_seq_test, labels_test)

In [21]:
# Create DataLoaders for training and testing
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

CNN Model

In [34]:
class CNN(nn.Module):
    def __init__(self, num_classes=8):  # Now predicting 8 classes for sst8
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, (5, 1), padding=(2, 0))
        self.p1 = nn.MaxPool2d((2, 1), (2, 1))
        self.conv2 = nn.Conv2d(32, 64, (5, 1), padding=(2, 0))
        self.p2 = nn.MaxPool2d((2, 1), (2, 1))
        self.flatten = nn.Flatten()
        self.d1 = nn.Linear(64 * (padded_seq.shape[1] // 4), 1024)  # Adjust for pooling
        self.dropout = nn.Dropout(0.2)
        self.d2 = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = x.float()
        x = x.squeeze()
        x = x.unsqueeze(1).unsqueeze(3)  # Add channel dimensions
        conv1 = torch.relu(self.conv1(x))
        p1 = self.p1(conv1)
        conv2 = torch.relu(self.conv2(p1))
        p2 = self.p2(conv2)
        flatten = self.flatten(p2)
        d1 = torch.relu(self.d1(flatten))
        dropout = self.dropout(d1)
        out = self.d2(dropout)
        out = out.unsqueeze(1).expand(-1, 4646, -1)
        return conv1, p1, conv2, p2, out
    
model = CNN()

In [23]:
# Loss func and Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

loss_object = nn.CrossEntropyLoss()

In [43]:
# Train and test steps
def train_step(images, labels):
    optimizer.zero_grad()  # Zero the gradients
    _, _, _, _, predictions = model(images)  # Forward pass
    valid_mask = labels != -1

    masked_predictions = predictions[valid_mask]  # Shape: [valid_seq_length, num_classes]
    masked_labels = labels[valid_mask]

    # loss = loss_object(predictions, labels)  # Compute the loss
    loss = loss_object(masked_predictions, masked_labels)  # Reshape for loss
    loss.backward()  # Backpropagation
    optimizer.step()  # Update weights
    #accuracy = (predictions.argmax(dim=1) == labels).float().mean().item()
    pred_labels = predictions.argmax(dim=2)
    masked_pred_labels = pred_labels[valid_mask]  # Masked predicted labels
    masked_true_labels = labels[valid_mask]
    accuracy = (masked_pred_labels.eq(masked_true_labels)).float().mean().item()  # Calculate accuracy
    return loss.item(), accuracy

def test_step(images, labels):
    _, _, _, _, predictions = model(images)  # Forward pass (no gradients)
    valid_mask = labels != -1

    reshaped_predictions = predictions.reshape(-1, 8)  # [N * seq_length, num_classes]
    reshaped_labels = labels.reshape(-1)

    t_loss = loss_object(reshaped_predictions[valid_mask.reshape(-1)], reshaped_labels[valid_mask.reshape(-1)])  # Compute test loss
    # accuracy = (predictions.argmax(dim=1) == labels).float().mean().item()
    pred_labels = predictions.argmax(dim=2)
    accuracy = (pred_labels[valid_mask].eq(labels[valid_mask])).float().mean().item()
    return t_loss.item(), accuracy

In [25]:
train_times = 0
test_times = 0

for images, labels in train_dataloader:
    train_times += 1
print("total train_times :" , train_times)

for images, labels in test_dataloader:
    test_times += 1
print("total test_times :" , test_times)


total train_times : 966
total test_times : 242


In [44]:
num_epochs = 20
test_acc = []

for epoch in range(num_epochs):
    train_loss = 0.0
    train_accuracy = 0.0
    test_loss = 0.0
    test_accuracy = 0.0
    train_steps = 0
    test_steps = 0
    print("epoch: ", epoch)

    # Training loop
    for images, labels in train_dataloader:
        images = images.unsqueeze(1).unsqueeze(3)  # Add dimensions for CNN input
        loss, accuracy = train_step(images, labels) 
        train_loss += loss
        train_accuracy += accuracy
        train_steps += 1
        print("train_steps done:" , train_steps)

    # Testing loop (no gradient calculations)
    with torch.no_grad():
        for test_images, test_labels in test_dataloader:
            test_images = test_images.unsqueeze(1).unsqueeze(3)  # Add dimensions for CNN input
            t_loss, t_accuracy = test_step(test_images, test_labels)
            test_loss += t_loss
            test_accuracy += t_accuracy
            test_steps += 1
            print("test_steps done:" , test_steps)

    test_acc.append(test_accuracy / test_steps)  # Save test accuracy for this epoch
    
    # Print results
    template = 'Epoch {}, Loss: {:.4f}, Accuracy: {:.4f}, Test Loss: {:.4f}, Test Accuracy: {:.4f}'
    print(template.format(epoch + 1,
                          train_loss / train_steps,
                          train_accuracy / train_steps,
                          test_loss / test_steps,
                          test_accuracy / test_steps))

epoch:  0
train_steps done: 1
train_steps done: 2
train_steps done: 3
train_steps done: 4
train_steps done: 5
train_steps done: 6
train_steps done: 7
train_steps done: 8
train_steps done: 9
train_steps done: 10
train_steps done: 11
train_steps done: 12
train_steps done: 13
train_steps done: 14
train_steps done: 15
train_steps done: 16
train_steps done: 17
train_steps done: 18
train_steps done: 19
train_steps done: 20
train_steps done: 21
train_steps done: 22
train_steps done: 23
train_steps done: 24
train_steps done: 25
train_steps done: 26
train_steps done: 27
train_steps done: 28
train_steps done: 29
train_steps done: 30
train_steps done: 31
train_steps done: 32
train_steps done: 33
train_steps done: 34
train_steps done: 35
train_steps done: 36
train_steps done: 37
train_steps done: 38
train_steps done: 39
train_steps done: 40
train_steps done: 41
train_steps done: 42
train_steps done: 43
train_steps done: 44
train_steps done: 45
train_steps done: 46
train_steps done: 47
train_steps 

KeyboardInterrupt: 